In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("MACHINE LEARNING PROJECT ")
df = pd.read_csv('breast_cancer_data.csv')


le = LabelEncoder()
df['Diagnosis'] = le.fit_transform(df['Diagnosis'])

X = df.drop('Diagnosis', axis=1)
y = df['Diagnosis']


print("\nDataset Information:")
print("Number of Samples: ", X.shape[0])
print("Number of Features: ", X.shape[1])
print("Class Labels: ", np.unique(y))
print("Number of Labels: ", len(y))
print("#Class 0 (Benign):", np.sum(y==0), " #Class 1 (malignant):", np.sum(y==1))

# Data splitting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)

print("\nSplit Information:")
print("Number of training_Samples: ", X_train.shape[0])
print("Number of testing_Samples: ", X_test.shape[0])
print("#Class 0 (Benign):", np.count_nonzero(y_test==0), " #Class 1 (Malignant):", np.count_nonzero(y_test==1))

print("\nClass distribution in training set:", np.bincount(y_train))
print("Class distribution in test set:", np.bincount(y_test))


sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

# Hyper parameter tuning of Models

from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

print("\nHyper Parameter information:")

param_lr = {'C': [0.1, 1, 10], 'solver': ['liblinear'], 'penalty': ['l1', 'l2']}
grid_lr = GridSearchCV(LogisticRegression(), param_lr, cv=5).fit(X_train_scaled, y_train)

param_Perceptron = {'alpha': [0.0001, 0.01], 'penalty': ['l1', 'l2', None], 'eta0': [0.1, 1.0]}
grid_Perceptron = GridSearchCV(Perceptron(), param_Perceptron, cv=5).fit(X_train_scaled, y_train)

param_dt = {'criterion': ['gini', 'entropy'], 'max_depth': [None, 5, 10], 'min_samples_split': [2, 5]}
grid_dt = GridSearchCV(DecisionTreeClassifier(), param_dt, cv=5).fit(X_train_scaled, y_train)

param_svm = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf'], 'gamma': ['scale', 'auto']}
grid_svm = GridSearchCV(SVC(), param_svm, cv=5).fit(X_train_scaled, y_train)

param_rf = {'n_estimators': [50, 100], 'max_features': ['sqrt', 'log2'], 'max_depth': [None, 10]}
grid_rf = GridSearchCV(RandomForestClassifier(), param_rf, cv=5).fit(X_train_scaled, y_train)

grids = [grid_lr, grid_Perceptron, grid_dt, grid_svm, grid_rf]
model_names = ['Logistic Regression', 'Perceptron', 'Decision Tree', 'SVM', 'Random Forest']

# Model Selection based on results
results = []
for i, grid in enumerate(grids):
    test_accuracy = grid.score(X_test_scaled, y_test)
    results.append({
        'ModelName': model_names[i],
        'ModelObject': grid.best_estimator_,
        'Test Accuracy': test_accuracy
    })


comparison_df = pd.DataFrame(results).sort_values(by='Test Accuracy', ascending=False)
best_model_object = comparison_df.iloc[0]['ModelObject']
best_model_name = comparison_df.iloc[0]['ModelName']


 # Feature Importance with visualization based on results
print(f"Best Model Name: {best_model_name}")
print(f"Best Model Object: {best_model_object}")


if hasattr(best_model_object, 'feature_importances_'):
    importances = best_model_object.feature_importances_
    feature_df = pd.DataFrame({
        'Feature': X.columns,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(
        x='Importance',
        y='Feature',
        data=feature_df.head(10),
        palette='viridis'
    )
    plt.title(f'Top 10 Most Important Features ({best_model_name})')
    plt.show()
else:
    print(f"\nNote: {best_model_name} is not  support feature importance visualization directly.")

# Prediction on test data
y_pred = best_model_object.predict(X_test_scaled)
# Evaluation on test (Accuracy, Precision, Recall, F1_score)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print("\n--- Final Evaluation Metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
# Plotting Confusion Matrix
plt.figure(figsize=(6, 4))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
plt.title(f'Confusion Matrix: {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()



MACHINE LEARNING PROJECT 


KeyError: 'Diagnosis'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv("breast_cancer_data.csv")

if "diagnosis" in df.columns:
    target_col = "diagnosis"
elif "Diagnosis" in df.columns:
    target_col = "Diagnosis"
elif "target" in df.columns:
    target_col = "target"
else:
    raise ValueError("Target column not found!")


if df[target_col].dtype == "object":
    le = LabelEncoder()
    df[target_col] = le.fit_transform(df[target_col])

X = df.drop(target_col, axis=1)
y = df[target_col]

print("Samples:", X.shape[0])
print("Features:", X.shape[1])
print("Class distribution:", np.bincount(y))

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)


param_gb = {   # Gradient Boosting
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 4]
}

grid_gb = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_gb,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_gb.fit(X_train, y_train)

best_gb = grid_gb.best_estimator_

print("\nBest Parameters:", grid_gb.best_params_)


y_pred = best_gb.predict(X_test)

print("\n--- Gradient Boosting Results ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred, zero_division=0):.4f}")


cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Benign", "Malignant"],
    yticklabels=["Benign", "Malignant"]
)
plt.title("Confusion Matrix - Gradient Boosting")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


feature_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_gb.feature_importances_
}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x="Importance", y="Feature", data=feature_df.head(10))
plt.title("Top 10 Important Features - Gradient Boosting")
plt.show()

